In [6]:
# ==========================================
# FINAL HACKATHON PROJECT: FOOD DELIVERY ANALYSIS (FIXED VERSION)
# ==========================================

import pandas as pd
import sqlite3
import os

print("🚀 STARTING ANALYSIS...\n")

# ---------------------------------------------------------
# STEP 1: LOAD DATASETS
# ---------------------------------------------------------
print("--- Loading Files ---")
# 1. Load Orders (CSV)
if os.path.exists('orders.csv'):
    df_orders = pd.read_csv('orders.csv')
    print("✓ Orders loaded")
else:
    print("❌ Error: orders.csv not found")

# 2. Load Users (JSON)
if os.path.exists('users.json'):
    df_users = pd.read_json('users.json')
    print("✓ Users loaded")
else:
    print("❌ Error: users.json not found")

# 3. Load Restaurants (SQL)
if os.path.exists('restaurants.sql'):
    conn = sqlite3.connect(':memory:')
    with open('restaurants.sql', 'r') as f:
        sql_script = f.read()
    conn.executescript(sql_script)
    df_restaurants = pd.read_sql("SELECT * FROM restaurants", conn)
    print("✓ Restaurants loaded")
else:
    print("❌ Error: restaurants.sql not found")

# ---------------------------------------------------------
# STEP 2: DATA MERGING & CLEANING
# ---------------------------------------------------------
print("\n--- Merging Data ---")

# Merge Orders + Users (Left Join on user_id)
df_merged = pd.merge(df_orders, df_users, on='user_id', how='left')

# Prepare Restaurants for Merge
# We try to rename 'name' to 'Restaurant_Name' if it exists in the SQL data
if 'name' in df_restaurants.columns:
    df_restaurants.rename(columns={'name': 'Restaurant_Name'}, inplace=True)
if 'city' in df_restaurants.columns:
    df_restaurants.rename(columns={'city': 'Restaurant_City'}, inplace=True)

# Merge with Restaurants (Left Join on restaurant_id)
final_df = pd.merge(df_merged, df_restaurants, on='restaurant_id', how='left')

# --- SMART COLUMN DETECTOR ---
# This block finds the correct columns even if names changed during merge
print("Detecting correct column names...")

# 1. Identify Total Amount Column
if 'total_amount' not in final_df.columns:
    # Sometimes it might be 'amount' or similar
    print("⚠️ 'total_amount' not found. Checking alternatives...")

final_df['total_amount'] = pd.to_numeric(final_df['total_amount'], errors='coerce')
final_df['rating'] = pd.to_numeric(final_df['rating'], errors='coerce')

# 2. Identify User City Column (Prefer user city, fallback to others)
city_col = 'city' # default
if 'city_x' in final_df.columns: city_col = 'city_x' # User city usually _x
elif 'city' in final_df.columns: city_col = 'city'
final_df['User_City'] = final_df[city_col]

# 3. Identify Restaurant Name Column
# It could be 'Restaurant_Name', 'restaurant_name', 'restaurant_name_x', etc.
possible_r_names = ['Restaurant_Name', 'restaurant_name', 'restaurant_name_x', 'name_y']
r_name_col = next((c for c in possible_r_names if c in final_df.columns), None)

if r_name_col:
    print(f"✓ Using '{r_name_col}' as Restaurant Name column.")
else:
    print("❌ CRITICAL: Could not find Restaurant Name column. Check input files.")
    r_name_col = 'restaurant_id' # Fallback to ID to prevent crash

# ---------------------------------------------------------
# STEP 3: SOLVING MCQ QUESTIONS
# ---------------------------------------------------------
print("\n📊 PART 1: MCQ SOLUTIONS")

# Q1: Highest Revenue City (Gold)
gold_orders = final_df[final_df['membership'] == 'Gold']
mcq1 = gold_orders.groupby('User_City')['total_amount'].sum().idxmax()
print(f"MCQ 1. Highest Revenue City (Gold): {mcq1}")

# Q2: Highest AOV Cuisine
mcq2 = final_df.groupby('cuisine')['total_amount'].mean().idxmax()
print(f"MCQ 2. Cuisine with Highest AOV: {mcq2}")

# Q3: Users > 1000 spend
user_totals = final_df.groupby('user_id')['total_amount'].sum()
mcq3 = (user_totals > 1000).sum()
print(f"MCQ 3. Users with >1000 spend: {mcq3}")

# Q4: Best Revenue Rating Range
def get_rating_range(r):
    if 3.0 <= r <= 3.5: return "3.0 – 3.5"
    if 3.6 <= r <= 4.0: return "3.6 – 4.0"
    if 4.1 <= r <= 4.5: return "4.1 – 4.5"
    if 4.6 <= r <= 5.0: return "4.6 – 5.0"
    return "Other"
final_df['rating_range'] = final_df['rating'].apply(get_rating_range)
mcq4 = final_df.groupby('rating_range')['total_amount'].sum().idxmax()
print(f"MCQ 4. Best Revenue Rating Range: {mcq4}")

# Q5: Highest AOV City (Gold)
mcq5 = gold_orders.groupby('User_City')['total_amount'].mean().idxmax()
print(f"MCQ 5. Highest AOV City (Gold): {mcq5}")

# Q6: Cuisine with fewest restaurants
cuisine_counts = final_df.groupby('cuisine')['restaurant_id'].nunique()
mcq6 = cuisine_counts.idxmin()
print(f"MCQ 6. Cuisine with fewest restaurants: {mcq6}")

# Q7: Gold Order %
mcq7 = (len(gold_orders) / len(final_df)) * 100
print(f"MCQ 7. Gold Order Percentage: {mcq7:.1f}%")

# Q8: High AOV Niche Restaurant (<20 orders)
# USES THE DETECTED COLUMN NAME HERE
rest_stats = final_df.groupby(r_name_col).agg({'total_amount':'mean', 'order_id':'count'})
mcq8 = rest_stats[rest_stats['order_id'] < 20]['total_amount'].idxmax()
print(f"MCQ 8. High AOV Niche Restaurant: {mcq8}")

# Q9: Best Revenue Combo
combos = [('Gold', 'Indian'), ('Gold', 'Italian'), ('Regular', 'Indian'), ('Regular', 'Chinese')]
best_rev = 0
best_combo = ""
for mem, cuis in combos:
    r = final_df[(final_df['membership'] == mem) & (final_df['cuisine'] == cuis)]['total_amount'].sum()
    if r > best_rev:
        best_rev = r
        best_combo = f"{mem} + {cuis}"
print(f"MCQ 9. Best Revenue Combo: {best_combo}")

# Q10: Best Revenue Quarter
date_col = [col for col in final_df.columns if 'date' in col.lower()][0]
final_df[date_col] = pd.to_datetime(final_df[date_col], dayfirst=True)
final_df['quarter'] = final_df[date_col].dt.quarter
mcq10 = final_df.groupby('quarter')['total_amount'].sum().idxmax()
print(f"MCQ 10. Best Revenue Quarter: Q{mcq10}")

# ---------------------------------------------------------
# STEP 4: NUMERICAL SOLUTIONS
# ---------------------------------------------------------
print("\n🔢 PART 2: NUMERICAL SOLUTIONS")

print(f"Num 1. Total Gold Orders: {len(gold_orders)}")

hyd_rev = final_df[final_df['User_City'] == 'Hyderabad']['total_amount'].sum()
print(f"Num 2. Hyderabad Revenue: {int(round(hyd_rev))}")

print(f"Num 3. Distinct Users: {final_df['user_id'].nunique()}")

print(f"Num 4. Gold Member AOV: {round(gold_orders['total_amount'].mean(), 2)}")

print(f"Num 5. High Rated Orders (>=4.5): {len(final_df[final_df['rating'] >= 4.5])}")

top_gold_city = gold_orders.groupby('User_City')['total_amount'].sum().idxmax()
num6 = len(gold_orders[gold_orders['User_City'] == top_gold_city])
print(f"Num 6. Orders in {top_gold_city} (Gold): {num6}")

# Save the final file just in case
final_df.to_csv('final_food_delivery_dataset.csv', index=False)
print("\n✅ ANALYSIS COMPLETE. File saved.")

🚀 STARTING ANALYSIS...

--- Loading Files ---
✓ Orders loaded
✓ Users loaded
✓ Restaurants loaded

--- Merging Data ---
Detecting correct column names...
✓ Using 'restaurant_name_x' as Restaurant Name column.

📊 PART 1: MCQ SOLUTIONS
MCQ 1. Highest Revenue City (Gold): Chennai
MCQ 2. Cuisine with Highest AOV: Mexican
MCQ 3. Users with >1000 spend: 2544
MCQ 4. Best Revenue Rating Range: 4.6 – 5.0
MCQ 5. Highest AOV City (Gold): Chennai
MCQ 6. Cuisine with fewest restaurants: Chinese
MCQ 7. Gold Order Percentage: 49.9%
MCQ 8. High AOV Niche Restaurant: Hotel Dhaba Multicuisine
MCQ 9. Best Revenue Combo: Gold + Italian
MCQ 10. Best Revenue Quarter: Q3

🔢 PART 2: NUMERICAL SOLUTIONS
Num 1. Total Gold Orders: 4987
Num 2. Hyderabad Revenue: 1889367
Num 3. Distinct Users: 2883
Num 4. Gold Member AOV: 797.15
Num 5. High Rated Orders (>=4.5): 3374
Num 6. Orders in Chennai (Gold): 1337

✅ ANALYSIS COMPLETE. File saved.
